In [ ]:
import os
import pandas as pd
import faiss
import numpy as np
from openai import OpenAI
import tiktoken

# 2️⃣ Pfad zum Ordner mit TXT/CSV-Dateien
FOLDER_PATH = r"..\\Hackathon\\data\\Klassifikationsdateien\\"
FOLDER_PATH = r"..\\Hackathon\\icd10_small\\"

# 3️⃣ Alle Dateien im Ordner durchgehen
all_chunks = []  # Liste für alle Code+Beschreibung

for filename in os.listdir(FOLDER_PATH):
    if filename.lower().endswith((".txt", ".csv")):
        filepath = os.path.join(FOLDER_PATH, filename)
        print(f"Lese Datei: {filename}")
        
        # CSV oder TXT
        if filename.lower().endswith(".csv"):
            df = pd.read_csv(filepath, dtype=str)
            # Annahme: Spalten heißen "Code" und "Beschreibung"
            for idx, row in df.iterrows():
                code = str(row.get("Code", "")).strip()
                desc = str(row.get("Beschreibung", "")).strip()
                if code and desc:
                    all_chunks.append(f"{code} - {desc}")
        else:  # einfache TXT-Datei
            with open(filepath, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line:
                        all_chunks.append(line)

print(f"Anzahl Chunks: {len(all_chunks)}")

Lese Datei: icd10gm2026syst_gruppen.txt
Anzahl Chunks: 11


In [70]:
all_chunks

['icd_code,krankheit,beschreibung,symptome,keywords,embedding_text',
 'C91.1,Chronische lymphatische Leukämie,Langsam verlaufende bösartige Erkrankung der B-Lymphozyten mit Ansammlung abnormaler Lymphozyten im Blut Knochenmark und lymphatischen Geweben.,Müdigkeit;Lymphozytose;Lymphknotenschwellung;Nachtschweiß,CLL;chronische Leukämie;B-Zell Malignom,"ICD C91.1 Chronische lymphatische Leukämie. Hämatologische Krebserkrankung mit Ansammlung reifer B-Lymphozyten. Typische Befunde sind erhöhte Lymphozyten im Blut vergrößerte Lymphknoten Müdigkeit und wiederkehrende Infektionen."',
 'C34.9,Lungenkarzinom nicht näher bezeichnet,Bösartiger Tumor des Lungengewebes häufig assoziiert mit Rauchen oder Umweltbelastungen.,anhaltender Husten;Bluthusten;Brustschmerz;Gewichtsverlust,Lungenkrebs;pulmonaler Tumor;Raucherkrankheit,"ICD C34.9 Lungenkarzinom. Maligne Neubildung der Lunge mit Symptomen wie chronischem Husten Brustschmerzen Bluthusten und Gewichtsverlust."',
 'C50.9,Mammakarzinom nicht näher

In [71]:
model = "text-embedding-3-small"
enc = tiktoken.encoding_for_model(model)

total_tokens = sum(len(enc.encode(t)) for t in all_chunks)
print(f"Gesamtanzahl Tokens: {total_tokens}")

Gesamtanzahl Tokens: 1254


In [73]:
# 4️⃣ Embeddings erzeugen
def get_embedding(text):
    resp = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return np.array(resp.data[0].embedding, dtype=np.float32)

embeddings = []
for chunk in all_chunks:
    emb = get_embedding(chunk)
    print(emb)
    embeddings.append(emb)

embeddings = np.vstack(embeddings)
print("Embeddings erstellt!")

np.save("icd10_embeddings.npy", embeddings)
np.save("icd10_chunks.npy", np.array(all_chunks))
print("Embeddings und Chunks gespeichert!")

[-0.03683472  0.0058403   0.01531219 ...  0.02429199 -0.00361252
 -0.00026345]
[ 0.01449585  0.0103302   0.03594971 ... -0.00016916 -0.02618408
 -0.00389862]
[ 0.00322342 -0.00604248  0.01841736 ... -0.02336121 -0.01885986
 -0.00517654]
[ 0.03335571  0.03927612 -0.00922394 ...  0.01490021 -0.00184917
 -0.00793457]
[ 0.00500107 -0.00600815  0.00875854 ...  0.00965118 -0.00994873
  0.00805664]
[-0.04391479 -0.00673294 -0.03671265 ...  0.0063591   0.03158569
  0.0100708 ]
[-0.02410889 -0.02322388 -0.00645065 ... -0.03549194 -0.00126266
 -0.00054502]
[-0.04211426  0.04190063  0.03421021 ...  0.0153656   0.00986481
 -0.01269531]
[ 0.02156067  0.02156067  0.02758789 ... -0.00477982 -0.02296448
  0.00796509]
[-0.04345703  0.02308655 -0.01939392 ...  0.0196991   0.01087952
  0.06536865]
[ 0.00882721  0.01596069  0.03210449 ... -0.01991272 -0.03320312
  0.0042572 ]
Embeddings erstellt!
Embeddings und Chunks gespeichert!


In [74]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
print(f"FAISS Index erstellt mit {index.ntotal} Einträgen.")

faiss.write_index(index, "icd10_faiss.index")
print("FAISS Index gespeichert!")

FAISS Index erstellt mit 11 Einträgen.
FAISS Index gespeichert!


In [75]:
# FAISS Index laden
index = faiss.read_index("icd10_faiss.index")
print(f"FAISS Index geladen mit {index.ntotal} Einträgen!")

# Chunks laden
all_chunks = list(np.load("icd10_chunks.npy", allow_pickle=True))

FAISS Index geladen mit 11 Einträgen!


In [ ]:
def retrieve(query, top_k=5):
    q_emb = get_embedding(query)
    D, I = index.search(np.array([q_emb]), top_k)
    results = [all_chunks[i] for i in I[0]]
    return results

def ask_rag(question, top_k=5):
    retrieved_chunks = retrieve(question, top_k)
    prompt = f"""
Du bist ein medizinisches Assistenzsystem. 
Nutze die folgenden ICD-10-Codes und Beschreibungen, um die Frage zu beantworten:

{retrieved_chunks}

Frage: {question}

Antwort (nur die gesamten relevante Codes und ihre Beschreibung. Keine weiteren Informationen dazu generieren):
"""
    response = client.chat.completions.create(
        model="gpt-5-mini",
        
        messages=[{"role": "user", "content": prompt}])
    return response.choices[0].message.content

# Test
frage = "Welcher ICD-10-Code nennt den Lendenwirbel?"
print(ask_rag(frage))

M54.5 — Kreuzschmerz: Häufige muskuloskelettale Beschwerde im Bereich der Lendenwirbelsäule.
